# Vivacity Integrity Gate

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
#!/usr/bin/env python3
"""Build Vivacity QA/inclusion gate before any causal modelling."""

from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


BASE = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
CUTOFF_DIR = BASE / "Vivacity_full_day_cutoff_20260526"
INPUT = CUTOFF_DIR / "control/vivacity_treated_and_control_daily_merged.csv"
OUT_DIR = CUTOFF_DIR / "integrity_gate"
QA_DIR = CUTOFF_DIR / "quality_checks"

INTERVENTION_ASSUMPTION = "exact scheme dates carried in the prepared input data"
MIN_WINDOW_DAYS = 21
MIN_RELIABLE_DAYS_IN_WINDOW = 14
AVAILABILITY_THRESHOLD = 80
PRAGMATIC_MIN_PRE_DAYS = 90
PRAGMATIC_MIN_POST_DAYS = 180
STRICT_MIN_PRE_DAYS = 365
STRICT_MIN_POST_DAYS = 365


def apply_confirmed_intervention_dates(df: pd.DataFrame) -> pd.DataFrame:
    if "intervention_date" not in df.columns:
        raise ValueError("Prepared input data must include an intervention_date column.")

    out = df.copy()
    out["intervention_date"] = pd.to_datetime(out["intervention_date"], errors="coerce")
    scheme_key = out["analysis_scheme_id"].astype(str).str.strip()
    missing_schemes = sorted(scheme_key[out["intervention_date"].isna()].dropna().unique().tolist())
    if missing_schemes:
        raise ValueError(
            "Prepared input data has missing intervention_date values for: "
            f"{missing_schemes}"
        )
    return out


def bool_series(df: pd.DataFrame, col: str, default: bool = False) -> pd.Series:
    if col not in df:
        return pd.Series(default, index=df.index)
    s = df[col]
    if s.dtype == bool:
        return s.fillna(default)
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def first_reliable_date(group: pd.DataFrame) -> pd.Timestamp | pd.NaT:
    g = group.sort_values("date").reset_index(drop=True)
    reliable = g["quality_reliable_day"].astype(bool).tolist()
    for i, val in enumerate(reliable):
        if not val:
            continue
        if i + MIN_WINDOW_DAYS > len(reliable):
            continue
        if sum(reliable[i : i + MIN_WINDOW_DAYS]) >= MIN_RELIABLE_DAYS_IN_WINDOW:
            return g.loc[i, "date"]
    return pd.NaT


def decision_reason(row: pd.Series, decision_col: str) -> str:
    reasons: list[str] = []
    if row["dataset_role"] == "treated":
        if row["treated_use_in_causal_models"]:
            return "Treated countline retained for scheme-side comparison after first reliable date."
        if pd.isna(row["first_reliable_date"]):
            reasons.append("no first reliable date identified")
        if row["usable_pre_days"] < PRAGMATIC_MIN_PRE_DAYS:
            reasons.append(f"usable pre-days below {PRAGMATIC_MIN_PRE_DAYS}")
        if row["usable_post_days"] < PRAGMATIC_MIN_POST_DAYS:
            reasons.append(f"usable post-days below {PRAGMATIC_MIN_POST_DAYS}")
        return "; ".join(reasons) or "treated countline excluded by quality rule"

    if row[decision_col]:
        if decision_col == "control_use_strict_causal":
            return "Control passes strict seasonal threshold: >=365 usable pre-days and >=365 usable post-days."
        return (
            "Control passes pragmatic exploratory threshold: >=90 usable pre-days, "
            ">=180 usable post-days, first reliable date before intervention."
        )

    if pd.isna(row["first_reliable_date"]):
        reasons.append("no first reliable date identified")
    if not row["has_reliable_before_intervention"]:
        reasons.append("first reliable date is not before confirmed intervention date")
    if row["usable_pre_days"] < PRAGMATIC_MIN_PRE_DAYS:
        reasons.append(f"usable pre-days below {PRAGMATIC_MIN_PRE_DAYS}")
    if row["usable_post_days"] < PRAGMATIC_MIN_POST_DAYS:
        reasons.append(f"usable post-days below {PRAGMATIC_MIN_POST_DAYS}")
    if row["data_error_days"] > 0:
        reasons.append("contains data-error days; these are excluded from analysis rows")
    if row["manual_site_check_required"]:
        reasons.append("manual check still needed to confirm this is not itself an intervention site")
    return "; ".join(reasons) or "control excluded by quality rule"


def main() -> int:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QA_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(INPUT, low_memory=False)
    df["date"] = pd.to_datetime(df["date"], format="mixed", errors="coerce")
    df = apply_confirmed_intervention_dates(df)
    df["active_travel_total"] = pd.to_numeric(df["active_travel_total"], errors="coerce").fillna(0)
    df["data_availability_percent_min"] = pd.to_numeric(df["data_availability_percent_min"], errors="coerce")
    df["availability_missing_bool"] = bool_series(df, "availability_missing", default=True) | df[
        "data_availability_percent_min"
    ].isna()
    df["low_availability_bool"] = bool_series(df, "low_availability", default=False) | (
        df["data_availability_percent_min"] < AVAILABILITY_THRESHOLD
    ).fillna(False)
    df["data_error_bool"] = bool_series(df, "data_error_any", default=False)
    df["structural_missing_bool"] = bool_series(df, "structural_missing_candidate", default=False)
    df["quality_ok_day"] = (
        df["date"].notna()
        & df["intervention_date"].notna()
        & ~df["availability_missing_bool"]
        & ~df["low_availability_bool"]
        & ~df["data_error_bool"]
    )
    df["quality_reliable_day"] = df["quality_ok_day"] & (df["active_travel_total"] > 0)

    key_cols = ["dataset_role", "analysis_scheme_id", "countline_id", "date"]
    duplicate_daily_keys = (
        df.groupby(key_cols, dropna=False).size().reset_index(name="rows_per_key").query("rows_per_key > 1")
    )
    duplicate_daily_keys.to_csv(OUT_DIR / "vivacity_integrity_duplicate_daily_keys.csv", index=False)

    first_rel = (
        df.groupby(["dataset_role", "analysis_scheme_id", "countline_id"], dropna=False)
        .apply(first_reliable_date, include_groups=False)
        .rename("first_reliable_date")
        .reset_index()
    )

    base_group_cols = [
        "dataset_role",
        "analysis_scheme_id",
        "scheme_id",
        "scheme_name",
        "road_group",
        "treated_road_group",
        "installation_month",
        "intervention_date",
        "countline_id",
        "countline_name",
        "route_type",
        "candidate_lsoa21cd",
        "candidate_lsoa21nm",
        "candidate_lad",
        "match_score",
        "geo_distance_km",
        "suggested_priority",
        "hardware_name",
    ]

    qa = (
        df.groupby(base_group_cols, dropna=False)
        .agg(
            raw_first_date=("date", "min"),
            raw_last_date=("date", "max"),
            total_daily_rows=("date", "size"),
            bad_date_rows=("date", lambda s: int(s.isna().sum())),
            availability_missing_days=("availability_missing_bool", "sum"),
            low_availability_days=("low_availability_bool", "sum"),
            data_error_days=("data_error_bool", "sum"),
            structural_missing_candidate_days=("structural_missing_bool", "sum"),
            zero_active_flow_days=("active_travel_total", lambda s: int((s <= 0).sum())),
            quality_ok_days=("quality_ok_day", "sum"),
            reliable_flow_days=("quality_reliable_day", "sum"),
            active_travel_total=("active_travel_total", "sum"),
        )
        .reset_index()
        .merge(first_rel, on=["dataset_role", "analysis_scheme_id", "countline_id"], how="left")
    )

    def add_period_counts(row: pd.Series) -> pd.Series:
        sub = df[
            (df["dataset_role"] == row["dataset_role"])
            & (df["analysis_scheme_id"].astype(str) == str(row["analysis_scheme_id"]))
            & (df["countline_id"] == row["countline_id"])
        ].copy()
        start = row["first_reliable_date"]
        if pd.isna(start):
            usable = sub.iloc[0:0]
        else:
            usable = sub[(sub["date"] >= start) & sub["quality_ok_day"]]
        pre = usable[usable["date"] < row["intervention_date"]]
        post = usable[usable["date"] >= row["intervention_date"]]
        return pd.Series(
            {
                "usable_days_after_first_reliable": int(len(usable)),
                "usable_pre_days": int(len(pre)),
                "usable_post_days": int(len(post)),
                "usable_pre_active_total": float(pre["active_travel_total"].sum()),
                "usable_post_active_total": float(post["active_travel_total"].sum()),
            }
        )

    period_counts = qa.apply(add_period_counts, axis=1)
    qa = pd.concat([qa, period_counts], axis=1)

    qa["has_first_reliable_date"] = qa["first_reliable_date"].notna()
    qa["has_reliable_before_intervention"] = qa["has_first_reliable_date"] & (
        qa["first_reliable_date"] < qa["intervention_date"]
    )
    qa["manual_site_check_required"] = qa["dataset_role"].eq("control")
    qa["treated_use_in_causal_models"] = qa["dataset_role"].eq("treated") & qa["has_reliable_before_intervention"] & (
        qa["usable_pre_days"] >= PRAGMATIC_MIN_PRE_DAYS
    ) & (qa["usable_post_days"] >= PRAGMATIC_MIN_POST_DAYS)
    qa["control_use_pragmatic_causal"] = qa["dataset_role"].eq("control") & qa[
        "has_reliable_before_intervention"
    ] & (qa["usable_pre_days"] >= PRAGMATIC_MIN_PRE_DAYS) & (
        qa["usable_post_days"] >= PRAGMATIC_MIN_POST_DAYS
    )
    qa["control_use_strict_causal"] = qa["dataset_role"].eq("control") & qa[
        "has_reliable_before_intervention"
    ] & (qa["usable_pre_days"] >= STRICT_MIN_PRE_DAYS) & (
        qa["usable_post_days"] >= STRICT_MIN_POST_DAYS
    )
    qa["control_use_descriptive_only"] = qa["dataset_role"].eq("control") & ~qa["control_use_pragmatic_causal"]

    qa["inclusion_tier"] = "treated_or_not_control"
    qa.loc[qa["dataset_role"].eq("treated") & qa["treated_use_in_causal_models"], "inclusion_tier"] = (
        "treated_valid_for_causal_side"
    )
    qa.loc[qa["dataset_role"].eq("treated") & ~qa["treated_use_in_causal_models"], "inclusion_tier"] = (
        "treated_quality_limited"
    )
    qa.loc[qa["control_use_strict_causal"], "inclusion_tier"] = "control_strict_seasonal_causal"
    qa.loc[
        qa["control_use_pragmatic_causal"] & ~qa["control_use_strict_causal"], "inclusion_tier"
    ] = "control_pragmatic_exploratory_causal"
    qa.loc[qa["control_use_descriptive_only"], "inclusion_tier"] = "control_descriptive_only_exclude_causal"

    qa["strict_decision_reason"] = qa.apply(lambda r: decision_reason(r, "control_use_strict_causal"), axis=1)
    qa["pragmatic_decision_reason"] = qa.apply(lambda r: decision_reason(r, "control_use_pragmatic_causal"), axis=1)

    qa = qa.sort_values(["analysis_scheme_id", "dataset_role", "inclusion_tier", "countline_id"])

    inclusion_path = OUT_DIR / "vivacity_qa_inclusion_table.csv"
    qa.to_csv(inclusion_path, index=False)

    control_decisions = qa[qa["dataset_role"].eq("control")].copy()
    control_decisions.to_csv(OUT_DIR / "vivacity_control_inclusion_decisions.csv", index=False)

    treated_decisions = qa[qa["dataset_role"].eq("treated")].copy()
    treated_decisions.to_csv(OUT_DIR / "vivacity_treated_quality_decisions.csv", index=False)

    summary = {
        "input": str(INPUT),
        "cutoff": "2026-05-26 inclusive",
        "intervention_date_assumption": INTERVENTION_ASSUMPTION,
        "quality_ok_day": (
            f"date present; intervention month present; data availability not missing; "
            f"dataAvailabilityPercent_min >= {AVAILABILITY_THRESHOLD}; no dataError"
        ),
        "first_reliable_date_rule": (
            f"first quality_ok day with active_travel_total > 0 and at least "
            f"{MIN_RELIABLE_DAYS_IN_WINDOW} reliable days in the following {MIN_WINDOW_DAYS}-day window"
        ),
        "pragmatic_causal_control_rule": (
            f"control with first reliable date before intervention, >= {PRAGMATIC_MIN_PRE_DAYS} usable pre-days, "
            f">= {PRAGMATIC_MIN_POST_DAYS} usable post-days"
        ),
        "strict_causal_control_rule": (
            f"control with first reliable date before intervention, >= {STRICT_MIN_PRE_DAYS} usable pre-days, "
            f">= {STRICT_MIN_POST_DAYS} usable post-days"
        ),
        "manual_site_check_required_for_all_controls": True,
        "all_countlines": int(len(qa)),
        "treated_countlines": int(qa["dataset_role"].eq("treated").sum()),
        "control_countlines": int(qa["dataset_role"].eq("control").sum()),
        "controls_pragmatic_causal": int(qa["control_use_pragmatic_causal"].sum()),
        "controls_strict_causal": int(qa["control_use_strict_causal"].sum()),
        "controls_descriptive_only": int(qa["control_use_descriptive_only"].sum()),
        "duplicate_daily_keys": int(len(duplicate_daily_keys)),
        "outputs": {
            "qa_inclusion_table": str(inclusion_path),
            "control_decisions": str(OUT_DIR / "vivacity_control_inclusion_decisions.csv"),
            "treated_decisions": str(OUT_DIR / "vivacity_treated_quality_decisions.csv"),
            "duplicate_daily_keys": str(OUT_DIR / "vivacity_integrity_duplicate_daily_keys.csv"),
        },
    }

    summary_path = OUT_DIR / "vivacity_integrity_gate_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    md = f"""# Vivacity Data Integrity Gate

Generated from `{INPUT}`.

## Cutoff and Date Assumptions

- Full-day cutoff: **2026-05-26 inclusive**.
- Intervention date: exact scheme date carried in the prepared input data.
- Rationale: exact scheme dates are now available and certain, so month-start installation assumptions should not be used for new modelling runs.

## Quality Rules

- `quality_ok_day`: date present, intervention month present, availability not missing, `dataAvailabilityPercent_min >= {AVAILABILITY_THRESHOLD}`, and no `dataError`.
- `first_reliable_date`: first `quality_ok_day` with `active_travel_total > 0` and at least {MIN_RELIABLE_DAYS_IN_WINDOW} reliable days in the following {MIN_WINDOW_DAYS}-day window.
- Usable pre/post days are counted only after `first_reliable_date` and only on `quality_ok_day` rows.

## Inclusion Rules

- Pragmatic exploratory causal control: first reliable date before intervention, at least {PRAGMATIC_MIN_PRE_DAYS} usable pre-days, and at least {PRAGMATIC_MIN_POST_DAYS} usable post-days.
- Strict seasonal causal control: first reliable date before intervention, at least {STRICT_MIN_PRE_DAYS} usable pre-days, and at least {STRICT_MIN_POST_DAYS} usable post-days.
- Controls failing the pragmatic rule are retained as descriptive-only controls, not causal controls.
- All controls still require manual site verification to ensure they are not themselves affected by active travel interventions.

## Gate Results

- Total countlines assessed: {summary['all_countlines']}
- Treated countlines: {summary['treated_countlines']}
- Control countlines: {summary['control_countlines']}
- Pragmatic causal controls: {summary['controls_pragmatic_causal']}
- Strict seasonal causal controls: {summary['controls_strict_causal']}
- Descriptive-only controls: {summary['controls_descriptive_only']}
- Duplicate daily keys: {summary['duplicate_daily_keys']}

## Outputs

- `vivacity_qa_inclusion_table.csv`: full treated/control QA and inclusion table.
- `vivacity_control_inclusion_decisions.csv`: control-only inclusion decisions.
- `vivacity_treated_quality_decisions.csv`: treated countline quality decisions.
- `vivacity_integrity_duplicate_daily_keys.csv`: duplicate key check.

## Modelling Instruction

Do not use controls marked `control_descriptive_only_exclude_causal` in the main causal comparison. Use `control_pragmatic_exploratory_causal` for exploratory causal comparison, and report that strict seasonal control coverage is limited.
"""
    (OUT_DIR / "vivacity_integrity_gate_exclusion_logic.md").write_text(md, encoding="utf-8")

    print(json.dumps(summary, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
